# Semana 14: Despliegue de Modelos (MLOps)
## Notebook de Ejercicios (NB2) – Despliegue y Monitoreo Simulado

**Propósito:** Implementar una API para un modelo, simular drift de datos y discutir estrategias de actualización.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Crear una API REST para un modelo de machine learning.
- Simular drift de datos modificando las entradas.
- Usar Evidently para detectar cambios en los datos.
- Discutir estrategias de actualización de modelos (A/B testing, reentrenamiento, etc.).

---

## 0. Configuración Inicial

Importamos las librerías necesarias y entrenamos un modelo simple (regresión logística con iris) que servirá como base para el despliegue.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import subprocess
import requests
import os
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Serialización
import joblib

# Evidently para detección de drift
try:
    from evidently import ColumnMapping
    from evidently.report import Report
    from evidently.metrics import DataDriftTable, ColumnDriftMetric
    from evidently.test_suite import TestSuite
    from evidently.tests import TestColumnDrift, TestNumberOfDriftedColumns
    EVIDENTLY_AVAILABLE = True
except ImportError:
    EVIDENTLY_AVAILABLE = False
    print("Evidently no está instalado. Se instalará más adelante.")

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Entrenamiento y Serialización del Modelo

Usaremos el dataset Iris para entrenar un modelo de clasificación simple.

In [ ]:
# Cargamos datos
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Entrenamos modelo
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# Evaluación
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy del modelo: {accuracy:.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=class_names))

# Guardamos el modelo con joblib
joblib.dump(model, 'modelo_iris.pkl')
print("\nModelo guardado como 'modelo_iris.pkl'")

---
## 2. Creación de la API con Flask

Creamos una API REST que expone el modelo para predicciones.

In [ ]:
# Contenido del archivo app.py
app_content = '''
import numpy as np
import joblib
from flask import Flask, request, jsonify

# Cargar modelo
model = joblib.load('modelo_iris.pkl')

# Crear app
app = Flask(__name__)

@app.route('/')
def home():
    return "API de predicción Iris. Use /predict con POST."

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Obtener datos del request
        data = request.get_json()
        
        # Convertir a array
        features = np.array(data['features']).reshape(1, -1)
        
        # Predecir
        prediction = model.predict(features)[0]
        probabilities = model.predict_proba(features)[0].tolist()
        
        # Mapear a nombre de clase
        class_names = ['setosa', 'versicolor', 'virginica']
        
        return jsonify({
            'prediction': int(prediction),
            'class_name': class_names[prediction],
            'probabilities': probabilities
        })
    except Exception as e:
        return jsonify({'error': str(e)})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

with open('app.py', 'w') as f:
    f.write(app_content)

print("Archivo app.py creado.")

In [ ]:
# Archivo requirements.txt
requirements_content = '''
flask==2.3.2
numpy==1.24.3
scikit-learn==1.3.0
joblib==1.3.1
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Archivo requirements.txt creado.")
!cat requirements.txt

### 2.1. Ejecutar la API en segundo plano

Iniciamos el servidor Flask para poder probar las predicciones.

In [ ]:
# Verificamos que no haya un proceso previo en el puerto 5000
!lsof -i:5000 || echo "Puerto 5000 libre"

# Iniciamos Flask en segundo plano
process = subprocess.Popen(['python', 'app.py'], 
                           stdout=subprocess.PIPE, 
                           stderr=subprocess.PIPE,
                           text=True)

# Esperamos a que el servidor inicie
time.sleep(5)
print("Servidor Flask iniciado en segundo plano.")

### 2.2. Probar la API localmente

In [ ]:
# Función para probar la API
def test_prediction(features):
    payload = {'features': features}
    try:
        response = requests.post('http://localhost:5000/predict', 
                                 json=payload,
                                 headers={'Content-Type': 'application/json'})
        if response.status_code == 200:
            return response.json()
        else:
            return {'error': f'Status code: {response.status_code}'}
    except Exception as e:
        return {'error': str(e)}

# Probamos con un ejemplo
test_features = [5.1, 3.5, 1.4, 0.2]  # setosa
result = test_prediction(test_features)
print("Resultado de la API:")
print(json.dumps(result, indent=2))

In [ ]:
# Probamos con varios ejemplos
for i in range(5):
    features = X_test[i].tolist()
    true_label = y_test[i]
    result = test_prediction(features)
    if 'prediction' in result:
        print(f"Ejemplo {i+1} - Real: {true_label} ({class_names[true_label]}), Predicho: {result['prediction']} ({result['class_name']})")
    else:
        print(f"Ejemplo {i+1} - Error: {result.get('error', 'Desconocido')}")

---
## 3. Instalación y Configuración de Evidently

Evidently es una herramienta open-source para monitorear y detectar drift en datos y modelos.

**Enlaces útiles:**
- Documentación oficial: [https://docs.evidentlyai.com/](https://docs.evidentlyai.com/)
- GitHub: [https://github.com/evidentlyai/evidently](https://github.com/evidentlyai/evidently)

In [ ]:
# Instalamos Evidently
!pip install evidently

In [ ]:
from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics import DataDriftTable, ColumnDriftMetric
from evidently.test_suite import TestSuite
from evidently.tests import TestColumnDrift, TestNumberOfDriftedColumns

print("Evidently importado correctamente.")

## 4. Preparación de Datos de Referencia y Actuales

Para detectar drift, necesitamos:
- **Datos de referencia**: generalmente los datos de entrenamiento.
- **Datos actuales**: datos nuevos que llegan al sistema (simularemos modificando los datos originales).

In [ ]:
# Datos de referencia (train)
reference_data = pd.DataFrame(X_train, columns=feature_names)
reference_data['target'] = y_train

# Datos actuales (simulamos nuevos datos)
# Usamos test como base y luego introducimos drift
current_data = pd.DataFrame(X_test, columns=feature_names)
current_data['target'] = y_test

print(f"Datos de referencia: {reference_data.shape}")
print(f"Datos actuales (base): {current_data.shape}")

# Configuración de columnas
column_mapping = ColumnMapping()
column_mapping.target = 'target'
column_mapping.prediction = None  # No tenemos predicciones aún
column_mapping.numerical_features = feature_names

### 4.1. Verificación sin drift (baseline)

Primero verificamos que no hay drift entre train y test (que deberían ser similares).

In [ ]:
# Crear reporte de data drift
data_drift_report = Report(metrics=[
    DataDriftTable(),
])

data_drift_report.run(reference_data=reference_data, current_data=current_data, column_mapping=column_mapping)

# Mostrar resultados
data_drift_report

In [ ]:
# Obtener métricas de drift
drift_result = data_drift_report.as_dict()
drift_share = drift_result['metrics'][0]['result']['share_of_drifted_columns']
print(f"Proporción de columnas con drift: {drift_share:.2%}")

if drift_share > 0.5:
    print("⚠️ Se detecta drift significativo en los datos.")
else:
    print("✅ No se detecta drift significativo.")

## 5. Simulación de Drift de Datos

Introducimos cambios artificiales en los datos actuales para simular drift.

In [ ]:
# Creamos una copia de los datos actuales y modificamos algunas columnas
drifted_data = current_data.copy()

# Aplicamos transformaciones para simular drift
# Ejemplo: desplazar la media de algunas características
drifted_data['sepal length (cm)'] = drifted_data['sepal length (cm)'] * 1.5 + 2
drifted_data['petal length (cm)'] = drifted_data['petal length (cm)'] * 1.2 + 1

# También podemos cambiar la distribución de la variable objetivo
# (opcional, pero aquí mantenemos target para simular datos reales)

print("Datos con drift creados.")
print(f"Media original sepal length: {current_data['sepal length (cm)'].mean():.2f}")
print(f"Media con drift sepal length: {drifted_data['sepal length (cm)'].mean():.2f}")

In [ ]:
# Visualizamos las distribuciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(current_data['sepal length (cm)'], alpha=0.5, label='Original', bins=20)
axes[0].hist(drifted_data['sepal length (cm)'], alpha=0.5, label='Con drift', bins=20)
axes[0].set_title('Distribución de sepal length')
axes[0].legend()

axes[1].hist(current_data['petal length (cm)'], alpha=0.5, label='Original', bins=20)
axes[1].hist(drifted_data['petal length (cm)'], alpha=0.5, label='Con drift', bins=20)
axes[1].set_title('Distribución de petal length')
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.1. Detectar drift con Evidently

In [ ]:
# Ejecutamos el reporte con los datos con drift
drift_report_drifted = Report(metrics=[
    DataDriftTable(),
])

drift_report_drifted.run(reference_data=reference_data, current_data=drifted_data, column_mapping=column_mapping)

drift_report_drifted

In [ ]:
# Extraemos métricas
drift_result_drifted = drift_report_drifted.as_dict()
drifted_columns = drift_result_drifted['metrics'][0]['result']['drifted_columns']
drift_share_drifted = drift_result_drifted['metrics'][0]['result']['share_of_drifted_columns']

print(f"Proporción de columnas con drift: {drift_share_drifted:.2%}")
print("\nColumnas con drift:")
for col, details in drifted_columns.items():
    if details['drift_detected']:
        print(f"  - {col}: p-value = {details['p_value']:.6f}")

### 5.2. Usando Test Suite para validación automática

In [ ]:
# Creamos un conjunto de pruebas
drift_test_suite = TestSuite(tests=[
    TestNumberOfDriftedColumns(),
    TestColumnDrift(column_name='sepal length (cm)'),
    TestColumnDrift(column_name='petal length (cm)'),
])

drift_test_suite.run(reference_data=reference_data, current_data=drifted_data, column_mapping=column_mapping)

drift_test_suite

In [ ]:
# Verificamos si las pruebas pasaron
test_results = drift_test_suite.as_dict()
for test in test_results['tests']:
    status = test['status']
    name = test['name']
    print(f"{name}: {status}")

---
## 6. Estrategias de Actualización de Modelos

Cuando se detecta drift, es necesario actualizar el modelo. Discutimos varias estrategias:

### 6.1. Reentrenamiento programado (Scheduled Retraining)

**Descripción:** Se reentrena el modelo periódicamente (ej. semanal, mensual) con los nuevos datos acumulados.

**Ventajas:**
- Simple de implementar.
- Asegura que el modelo se actualiza regularmente.

**Desventajas:**
- Puede ser ineficiente si no hay cambios significativos.
- No responde rápidamente a drift repentino.

**Implementación:**
```python
# Pseudocódigo
if current_date == retrain_day:
    new_data = load_new_data()
    model.fit(all_data)
    save_model(model)
```

### 6.2. Reentrenamiento bajo demanda (Trigger-based Retraining)

**Descripción:** Se reentrena el modelo cuando se detecta drift o cuando el rendimiento cae por debajo de un umbral.

**Ventajas:**
- Eficiente (solo se reentrena cuando es necesario).
- Responde rápidamente a cambios.

**Desventajas:**
- Requiere monitoreo continuo.
- Puede ser reactivo (el modelo ya está desactualizado cuando se detecta).

**Implementación:**
```python
# Pseudocódigo
drift_detected = check_drift(reference_data, current_data)
if drift_detected:
    new_model = train_model(current_data)
    deploy_model(new_model)
```

### 6.3. A/B Testing

**Descripción:** Se despliega un nuevo modelo en paralelo con el actual y se compara su rendimiento en producción antes de reemplazarlo completamente.

**Ventajas:**
- Permite validar el nuevo modelo en condiciones reales.
- Reduce el riesgo de desplegar un modelo peor.

**Desventajas:**
- Requiere infraestructura para enrutar tráfico.
- Mayor complejidad operativa.

**Implementación:**
```python
# Pseudocódigo
if user_id % 2 == 0:
    prediction = model_a.predict(features)
else:
    prediction = model_b.predict(features)

# Después de recopilar suficientes datos, comparar métricas
if model_b_metrics > model_a_metrics:
    deploy_model_b()
```

### 6.4. Aprendizaje continuo (Online Learning)

**Descripción:** El modelo se actualiza incrementalmente con cada nuevo dato (ej. SGD, River).

**Ventajas:**
- Se adapta continuamente.
- No requiere reentrenamiento completo.

**Desventajas:**
- No todos los modelos soportan aprendizaje online.
- Puede ser sensible al orden de los datos.

**Ejemplo con SGDClassifier:**
```python
from sklearn.linear_model import SGDClassifier
model = SGDClassifier()
model.partial_fit(X_batch, y_batch, classes=[0,1,2])
```

### 6.5. Blue-Green Deployment y Canary Releases

**Blue-Green Deployment:**
- Se mantienen dos entornos (blue = actual, green = nuevo).
- Se valida el green y luego se cambia el tráfico completamente.

**Canary Releases:**
- Se despliega el nuevo modelo a un pequeño porcentaje de usuarios.
- Se monitorea y gradualmente se aumenta el porcentaje.

**Herramientas:** Kubernetes, Istio, AWS CodeDeploy.

### 6.6. Monitoreo continuo

Además de drift en datos, es importante monitorear:

1. **Rendimiento del modelo**: Accuracy, precisión, recall en producción (si se tienen etiquetas).
2. **Drift en predicciones**: Distribución de las salidas del modelo.
3. **Latencia y uso de recursos**: Tiempo de respuesta, CPU, memoria.
4. **Data quality**: Valores nulos, outliers, etc.

**Herramientas:**
- Evidently (como vimos).
- Prometheus + Grafana.
- MLflow.
- WhyLabs.

### 6.7. Ejemplo simulado de reentrenamiento

Simulamos un pipeline de detección de drift y reentrenamiento.

In [ ]:
# Función para detectar drift
def detect_significant_drift(reference, current, threshold=0.3):
    report = Report(metrics=[DataDriftTable()])
    report.run(reference_data=reference, current_data=current, column_mapping=column_mapping)
    result = report.as_dict()
    drift_share = result['metrics'][0]['result']['share_of_drifted_columns']
    return drift_share > threshold, drift_share

# Simulamos un proceso de monitoreo
print("=== Monitoreo de drift en producción ===\n")

# Primera evaluación (sin drift)
drift_detected, share = detect_significant_drift(reference_data, current_data)
print(f"Evaluación 1 - Drift share: {share:.2%}")
if drift_detected:
    print("⚠️ Drift detectado. Se recomienda reentrenar.")
else:
    print("✅ Modelo actual OK.")

print()

# Segunda evaluación (con drift)
drift_detected, share = detect_significant_drift(reference_data, drifted_data)
print(f"Evaluación 2 - Drift share: {share:.2%}")
if drift_detected:
    print("⚠️ Drift detectado. Se recomienda reentrenar.")
    
    # Simulamos reentrenamiento con nuevos datos
    print("\n--- Iniciando reentrenamiento ---")
    # Combinamos datos de referencia y nuevos para entrenar
    X_retrain = np.vstack([X_train, drifted_data[feature_names].values])
    y_retrain = np.hstack([y_train, drifted_data['target'].values])
    
    new_model = LogisticRegression(max_iter=200, random_state=42)
    new_model.fit(X_retrain, y_retrain)
    
    # Evaluamos nuevo modelo
    new_accuracy = accuracy_score(y_test, new_model.predict(X_test))
    print(f"Nuevo modelo entrenado. Accuracy: {new_accuracy:.4f}")
    print("--- Reentrenamiento completado ---")
else:
    print("✅ Modelo actual OK.")

---
## 7. Recursos Adicionales y Herramientas

### Herramientas de MLOps:

| Herramienta | Propósito | Enlace |
|------------|-----------|--------|
| **MLflow** | Registro de experimentos, empaquetado, despliegue | [https://mlflow.org/](https://mlflow.org/) |
| **Evidently** | Monitoreo de drift y calidad de datos | [https://evidentlyai.com/](https://evidentlyai.com/) |
| **Docker** | Contenerización | [https://www.docker.com/](https://www.docker.com/) |
| **Kubernetes** | Orquestación de contenedores | [https://kubernetes.io/](https://kubernetes.io/) |
| **Prometheus** | Monitoreo de métricas | [https://prometheus.io/](https://prometheus.io/) |
| **Grafana** | Visualización de métricas | [https://grafana.com/](https://grafana.com/) |
| **KServe** | Serving de modelos en Kubernetes | [https://kserve.github.io/](https://kserve.github.io/) |
| **BentoML** | Empaquetado y despliegue | [https://www.bentoml.com/](https://www.bentoml.com/) |
| **Ray Serve** | Serving escalable | [https://docs.ray.io/en/latest/serve/](https://docs.ray.io/en/latest/serve/) |

### Conceptos clave de MLOps:

- **CI/CD para ML**: Automatización de entrenamiento, pruebas y despliegue.
- **Feature Store**: Repositorio centralizado de características.
- **Model Registry**: Registro de versiones de modelos.
- **Shadow Mode**: Despliegue de un nuevo modelo en paralelo sin afectar decisiones.
- **A/B Testing**: Comparación de modelos en producción.
- **Canary Deployment**: Despliegue gradual.
- **Rollback**: Capacidad de volver a versión anterior.

### Lecturas recomendadas:

- [Machine Learning Engineering in Production (Coursera)](https://www.coursera.org/specializations/machine-learning-engineering-for-production-mlops)
- [Designing Machine Learning Systems (O'Reilly)](https://www.oreilly.com/library/view/designing-machine-learning/9781098107956/)
- [MLOps: Continuous delivery and automation pipelines in machine learning](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)

---
## 8. Detener la API

Finalmente, detenemos el servidor Flask.

In [ ]:
process.terminate()
print("Servidor Flask detenido.")

---
## 9. Conclusiones

En este notebook hemos explorado el ciclo de vida de un modelo en producción:

✔️ **API REST**: Creamos y probamos una API para el modelo Iris.

✔️ **Detección de drift**:
- Instalamos y configuramos Evidently.
- Comparamos datos de referencia vs actuales.
- Simulamos drift y lo detectamos automáticamente.

✔️ **Estrategias de actualización**:
- Reentrenamiento programado vs bajo demanda.
- A/B testing, canary releases, blue-green deployment.
- Aprendizaje continuo.

✔️ **Monitoreo**: Importancia de monitorear datos, predicciones y rendimiento.

✔️ **Herramientas MLOps**: Panorama de herramientas disponibles.

**Lección clave**: Un modelo en producción no es estático; requiere monitoreo continuo y estrategias de actualización para mantener su rendimiento a lo largo del tiempo.

---
**Fin del Notebook de Ejercicios - Semana 14**
**Fin del Curso de Machine Learning**